<h1>Chapter 7 - Evaluation</h1>
<i>Measuring whether your `TinyAgent` actually works.</i>


<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents/blob/main/chapter07/chapter07.ipynb)

---

This notebook is for Chapter 7 of [An Illustrated Guide to AI Agents](https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ) by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>

### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter.

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to **Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM - `Gemma 4`

We use the same LLM that we used previously, namely the Gemma 4 E4B model with native tool calling and reasoning capabilities. Note that it will also be used to judge itself, as we will explore in the LLM-as-a-judge section.

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 4 E4B (with native thinking and tool calling)
llm = LLM(model="gemma4:e4b", think=True)

If you want to use another LLM, here are a couple of options (both locally and on the cloud) that you can try:

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-4-e4B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-4-e4b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-4-31b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2 - Building the `Evaluator`

We start by building a two small classes that allow you to do a straightforward evaluation, the `Benchmark` and the `Evaluator`:

In [ ]:
from dataclasses import dataclass
from typing import Callable

# Type hint for the scorers
##  (prediction: str, example: dict) -> bool | float
Scorer = Callable[[str, dict], bool | float]

@dataclass
class Benchmark:
    name: str
    examples: list[dict]
    scorer: Callable

In [ ]:
from illustrated_agents.chapters.ch4 import Memory
from illustrated_agents.chapters.ch5 import NativeTools
from illustrated_agents.chapters.ch6 import NativeReAct, TinyAgent


class Evaluator:
    """Run a TinyAgent over a Benchmark and aggregate the results."""

    def __init__(self, create_agent: Callable):
        """Initialize with a function that creates a new agent instance."""
        self.create_agent = create_agent

    def run(self, benchmark: Benchmark) -> dict:
        """Run the agent on each example in the benchmark and score the results."""

        # Run each example and collect results
        results = []
        for example in benchmark.examples:
            agent = self.create_agent()
            prediction = agent.run(example["task"]) or ""
            passed = benchmark.scorer(prediction, example)
            results.append(
                {
                    "prediction": prediction,
                    "passed": passed,
                }
            )

        # Aggregate pass rate
        if results:
            pass_rate = sum(result["passed"] for result in results) / len(results)
        else:
            pass_rate = 0.0

        # Return detailed results and overall pass rate
        return {
            "name": benchmark.name,
            "pass_rate": pass_rate,
            "results": results,
        }


def create_agent():
    """Create a new instance of TinyAgent"""
    return TinyAgent(
        llm=llm,
        memory=Memory(),
        tools=NativeTools(),
        planner=NativeReAct(),
    )

The `create_agent` allows us to create a new instance of `TinyAgent` whenever you run a new benchmark. This essentially serves as a way to hit reset on its memory and start from scratch. 

## 3 - Exact Match

The exact match scorer is a good way to start off. It is very strict but as a result allows you evaluate behavior that should not diverge in any way from what you want.
We're using MMLU-Pro, which consists of many different tasks with multiple-choice answers and is a great benchmark to start with.

In [ ]:
import re

def exact_match_scorer(prediction: str, example: dict) -> bool:
    """Return True if the answer matches the prediction, False otherwise"""
    match = re.search(r"\b([A-J])\b", prediction.upper())
    return match.group(1) == example["expected"]

In [ ]:
# Three examples from MMLU Pro
mmlu_pro = Benchmark(
    name="MMLU Pro",
    examples=[
        {
            "task": """Which of the following is the body cavity that contains the pituitary gland?
A) Ventral B) Dorsal C) Buccal D) Thoracic E) Pericardial F) Abdominal G) Spinal H) Pelvic I) Pleural J) Cranial
Answer with only the letter.""",
            "expected": "J",
        },
        {
            "task": """What is the approximate mean cranial capacity of Homo erectus?
A) 1200 cc B) under 650 cc C) 1700 cc D) 1350 cc E) just under 1000 cc F) 1500 cc G) under 500 cc H) about 800 cc I) just over 1100 cc J) about 900 cc
Answer with only the letter.""",
            "expected": "E",
        },
        {
            "task": """	
According to Moore’s “ideal utilitarianism,” the right action is the one that brings about the greatest amount of:
A) wealth. B) virtue. C) fairness. D) pleasure. E) peace. F) justice. G) happiness. H) power. I) good. J) knowledge.
Answer with only the letter.""",
            "expected": "I",
        },
    ],
    scorer=exact_match_scorer,
)

In [ ]:
from rich import print

# Run evaluation
result = Evaluator(create_agent).run(mmlu_pro)
print(result)

## 4 - Programmatic Check

The programmatic check is an interesting one since it allows you to essentially create many different types of checks in one. Here, examples are shown of the IFeval benchmark that has various rules on what the output should be. 

In [ ]:
def programmatic_scorer(prediction: str, example: dict) -> bool:
    """Check a prediction against its related check."""
    return example["check"](prediction)

In [ ]:
# Three examples from IFeval
ifeval = Benchmark(
    name="IFeval",
    examples=[
        {
            "task": "Write me a funny song with less than 10 sentences for a proposal to build a new playground at my local elementary school.",
            "check": lambda text: sum(1 for c in text if c in ".!?") < 10,
        },
        {
            "task": "Write an ad copy for a new product, a digital photo frame that connects to your social media accounts and displays your photos. Respond with at most 150 words.",
            "check": lambda text: len(text.split()) <= 150,
        },
        {
            "task": "I am planning a trip to Japan, and I would like thee to write an itinerary for my journey in a Shakespearean style. You are not allowed to use any commas in your response.",
            "check": lambda text: "," not in text,
        },
    ],
    scorer=programmatic_scorer,
)

In [ ]:
# Run evaluation
result = Evaluator(create_agent).run(ifeval)
print(result)

## 5 - LLM-as-a-judge

Using another LLM as a judge is a great way to evaluate the quality of a smaller LLM. 

In [ ]:
judge = LLM(
    model="gemini-3.1-flash-lite", 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai", 
    api_key="MY_API_KEY"
)

In [ ]:
def judge_scorer(prediction: str, example: dict) -> bool:
    """The LLM-as-a-judge scorer."""
    prompt = f"""
Score the response from 0.0 to 1.0.

Expected: {example["expected"]}
Response: {prediction}

Reply with only a single number.
"""
    response = judge.generate([{"role": "user", "content": prompt}])
    score = float(response.content.strip().split()[0])
    return score

In [ ]:
# Three examples adapted from MMLU Pro
mmlu_pro = Benchmark(
    name="MMLU Pro",
    examples=[
        {
            "task": "Which body cavity contains the pituitary gland?",
            "expected": "the cranial cavity",
        },
        {
            "task": "What is the approximate mean cranial capacity of Homo erectus?",
            "expected": "just under 1000 cc",
        },
        {
            "task": "According to Moore's 'ideal utilitarianism,' the right action is the one that brings about the greatest amount of what?",
            "expected": "good",
        },
    ],
    scorer=judge_scorer,
)

In [ ]:
# Run evaluation
result = Evaluator(create_agent).run(mmlu_pro)
print(result)

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

We added a single new module — `evaluator.py` — on top of the Chapter 6 agent. The agent itself did not change. That separation is the point: a clean boundary between the system under test and the system doing the testing lets you iterate on either side without breaking the other.

In [ ]:
from illustrated_agents.chapters.ch7 import what_we_built; what_we_built

# What's Next

Up next is Chapter 8 on Multi-Agent Collaboration.